# Khám phá Dữ liệu Chuyên sâu (EDA) - Sub-task 3.1
Notebook này thực hiện việc kết nối với Supabase, kéo dữ liệu từ View `mv_bi_mart_hourly_measures` kết hợp với bảng `dim_solar_site`, kiểm tra chất lượng và thống kê mô tả.

In [ ]:
import os
import logging
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import urllib.parse 

# 1. Cấu hình Logging (Vẫn in ra console của Jupyter cho dễ theo dõi)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# 2. Load biến môi trường và kết nối
load_dotenv()
db_user = os.getenv("DB_USER")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
raw_password = os.getenv("DB_PASSWORD")
db_password = urllib.parse.quote_plus(raw_password) if raw_password else None

if not all([db_user, db_password, db_host, db_port, db_name]):
    logger.error("Không tìm thấy đủ các biến môi trường trong file .env!")
else:
    DATABASE_URL = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    try:
        engine = create_engine(DATABASE_URL)
        logger.info("Đã kết nối thành công với Supabase.")
    except Exception as e:
        logger.critical(f"Lỗi kết nối database: {e}")

In [ ]:
# 3. Kéo dữ liệu từ BI Mart
query = """
    SELECT 
        m.*, 
        s.capacity_kw 
    FROM bi_mart.mv_bi_mart_hourly_measures m
    LEFT JOIN bi_mart.dim_solar_site s ON m.site_id = s.site_id
"""
logger.info("Đang thực thi truy vấn kéo dữ liệu...")
df = pd.read_sql_query(query, engine)

if df.empty:
    logger.error("View trả về 0 dòng! Cần kiểm tra lại DWH.")
else:
    logger.info(f"Đã kéo thành công {df.shape[0]} dòng và {df.shape[1]} cột.")
    
    # QA/QC Check
    missing_data = df.isnull().sum()
    total_missing = missing_data.sum()
    
    if total_missing == 0:
        logger.info("Chất lượng dữ liệu: ĐẠT.")
    else:
        logger.warning(f"Phát hiện {total_missing} giá trị bị thiếu!")
        display(pd.DataFrame(missing_data[missing_data > 0], columns=['Số lượng missing']))

In [ ]:
# 4. Thống kê mô tả toàn bộ
logger.info("BẢNG THỐNG KÊ MÔ TẢ TỔNG THỂ")

# Chọn cột số và tính thống kê
df_numeric = df.select_dtypes(include=['number'])
thong_ke_mo_ta = df_numeric.describe().T.round(2)

# Hiển thị bảng dạng HTML đẹp mắt trong Jupyter
display(thong_ke_mo_ta)

In [ ]:
# 5. Thống kê tập trung vào site có capacity_kw
cap_col = 'capacity_kw' 

if cap_col in df.columns:
    df_capacity_not_null = df[df[cap_col].notnull()]
    
    if not df_capacity_not_null.empty:
        logger.info(f"BẢNG THỐNG KÊ MÔ TẢ (Chỉ tính {df_capacity_not_null.shape[0]} dòng có {cap_col})")
        
        df_not_null_numeric = df_capacity_not_null.select_dtypes(include=['number'])
        thong_ke_mo_ta_not_null = df_not_null_numeric.describe().T.round(2)
        
        display(thong_ke_mo_ta_not_null)
    else:
        logger.warning(f"Cảnh báo: Toàn bộ dữ liệu đều bị null ở cột {cap_col}!")
else:
    logger.error(f"Không tìm thấy cột '{cap_col}'!")